## 1️⃣ Environment Setup & GPU Verification

In [23]:
# ============================================================
# 1.1 Verify GPU is available
# ============================================================
import subprocess

print("=" * 70)
print("🔍 Checking GPU availability...")
print("=" * 70)

# Check NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print("❌ ERROR: No NVIDIA GPU detected!")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
    raise RuntimeError("GPU not available")

print("\n✅ GPU detected! Proceeding with setup...")

🔍 Checking GPU availability...
Tue Jan  6 16:46:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             61W /  400W |    1469MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------

In [24]:
# ============================================================
# 1.2 Install dependencies (Colab with TF 2.19.0)
# ============================================================
# ⚠️ If you see numpy/h5py errors, go to Runtime > Restart runtime
# then run cells 1 and 2 again (skip pip install, just verify)

print("📦 Checking/Installing dependencies...")

# Check if we need to install anything
import subprocess
result = subprocess.run(['pip', 'show', 'xgboost'], capture_output=True, text=True)
if 'not found' in result.stderr.lower() or result.returncode != 0:
    print("   Installing additional packages...")
    !pip install -q xgboost>=2.0.3 rich>=13.7.1 python-dotenv>=1.0.0 structlog>=24.1.0
else:
    print("   ✓ Packages already installed")

# Verify key packages
import tensorflow as tf
import numpy as np
import pandas as pd

print(f"\n✅ Dependencies ready!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   GPU: {tf.config.list_physical_devices('GPU')}")

📦 Checking/Installing dependencies...
   ✓ Packages already installed

✅ Dependencies ready!
   TensorFlow: 2.19.0
   NumPy: 1.26.4
   Pandas: 2.2.0
   GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [45]:
# ============================================================
# 1.3 Verify TensorFlow CUDA setup & Optimize for A100
# ============================================================
import tensorflow as tf

print("=" * 70)
print("🔧 TensorFlow Configuration")
print("=" * 70)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

# Enable memory growth to prevent OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✅ Memory growth enabled for {len(gpus)} GPU(s)")
        
        # Get GPU details
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        gpu_name = gpu_details.get('device_name', 'Unknown')
        print(f"   GPU: {gpu_name}")
        
        # A100-specific optimizations
        if 'A100' in gpu_name:
            print(f"\n🚀 A100 detected! Enabling optimizations:")
            print(f"   • 80GB VRAM - using batch_size=512")
            print(f"   • TF32 enabled for matmul (faster on A100)")
            # Enable TF32 for A100 (faster than FP32, same accuracy)
            tf.config.experimental.enable_tensor_float_32_execution(True)
    except RuntimeError as e:
        print(f"⚠️ Could not set memory growth: {e}")

# ⚠️ MIXED PRECISION DISABLED - causes 0% accuracy bug in TF 2.19
# The accuracy metric fails to properly threshold float16 sigmoid outputs
# TF32 alone provides ~1.5x speedup without this issue
# tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f"\n✅ Using float32 precision (TF32 enabled for A100)")
print("   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug")

🔧 TensorFlow Configuration
TensorFlow version: 2.19.0
CUDA available: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
⚠️ Could not set memory growth: Physical devices cannot be modified after being initialized

✅ Using float32 precision (TF32 enabled for A100)
   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug


## 2️⃣ Clone Repository & Setup

In [53]:
# ============================================================
# 2.1 Clone the ML Engine repository
# ============================================================
# ⚠️ RE-RUN THIS CELL if you see dimension mismatch errors!
# This pulls the latest code with bug fixes from GitHub.

import os

REPO_URL = "https://github.com/Raynergy-svg/ml_engine.git"
REPO_DIR = "/content/ml_engine"

# IMPORTANT: Reset to /content first (fixes "getcwd" errors after rm -rf)
os.chdir("/content")

# Remove existing directory if it exists (forces fresh clone)
if os.path.exists(REPO_DIR):
    print("🗑️ Removing existing repo to get latest fixes...")
    !rm -rf {REPO_DIR}

print(f"📥 Cloning repository from {REPO_URL}...")
!git clone {REPO_URL} {REPO_DIR}

# Change to repo directory
os.chdir(REPO_DIR)
print(f"\n📂 Working directory: {os.getcwd()}")

# Show latest commit to verify we have the fix
print("\n📋 Latest commit:")
!git log --oneline -3

print("\n📁 Repository contents:")
!ls -la

🗑️ Removing existing repo to get latest fixes...
📥 Cloning repository from https://github.com/Raynergy-svg/ml_engine.git...
Cloning into '/content/ml_engine'...
remote: Enumerating objects: 1658, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 1658 (delta 69), reused 66 (delta 56), pack-reused 1570 (from 3)
Receiving objects: 100% (1658/1658), 352.32 MiB | 15.25 MiB/s, done.
Resolving deltas: 100% (833/833), done.
Updating files: 100% (504/504), done.

📂 Working directory: /content/ml_engine

📋 Latest commit:
ae2e7f5 (HEAD -> main, origin/main, origin/HEAD) fix: Handle threshold=0 for direction labeling to reduce class imbalance
0b9e925 fix: Add replay buffer clearing and show git log in clone cell
bfc382a fix: Handle replay buffer feature dimension mismatch

📁 Repository contents:
total 72612
drwxr-xr-x 15 root root     4096 Jan  6 17:26  .
drwxr-xr-x  1 root root     4096 Jan  6 17:25  ..
-rw-r--r--  1 root root    18

In [54]:
# ============================================================
# 2.2 Create necessary directories & Clear stale replay buffers
# ============================================================
import os
import glob
from pathlib import Path

directories = [
    "trained_data/models",
    "trained_data/checkpoints",
    "trained_data/checkpoints/tensorflow",
    "trained_data/replay/EUR_USD",
    "trained_data/replay/USD_JPY",
    "trained_data/replay/GBP_USD",
    "trained_data/logs",
    "trained_data/scalers",
    "market_data",
]

for d in directories:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {d}")

# Clear any stale replay buffers (they may have different feature dimensions)
replay_files = glob.glob("trained_data/replay/**/*.npz", recursive=True)
if replay_files:
    print(f"\n🗑️ Clearing {len(replay_files)} stale replay buffer(s)...")
    for f in replay_files:
        os.remove(f)
        print(f"   Removed: {f}")
    print("✅ Replay buffers cleared (prevents feature dimension mismatch)")
else:
    print("\n✅ No stale replay buffers to clear")

print("\n📁 Directory structure ready!")

✅ Created: trained_data/models
✅ Created: trained_data/checkpoints
✅ Created: trained_data/checkpoints/tensorflow
✅ Created: trained_data/replay/EUR_USD
✅ Created: trained_data/replay/USD_JPY
✅ Created: trained_data/replay/GBP_USD
✅ Created: trained_data/logs
✅ Created: trained_data/scalers
✅ Created: market_data

🗑️ Clearing 2 stale replay buffer(s)...
   Removed: trained_data/replay/USD_JPY/buffer.npz
   Removed: trained_data/replay/EUR_USD/buffer.npz
✅ Replay buffers cleared (prevents feature dimension mismatch)

📁 Directory structure ready!


## 3️⃣ Environment Variables (OANDA API)

In [9]:
# ============================================================
# 3.1 Set OANDA API credentials
# ============================================================
import os
from getpass import getpass

print("=" * 70)
print("🔐 OANDA API Configuration")
print("=" * 70)
print("\nEnter your OANDA Practice account credentials.")
print("(These are stored only in this session's memory)\n")

# Interactive input with clear format hints
print("API Token format: xxxx-xxxx (long string with hyphen)")
OANDA_API_TOKEN = getpass("OANDA API Token: ")

print("\nAccount ID format: 101-001-XXXXXXXX-001")
OANDA_ACCOUNT_ID = input("OANDA Account ID: ")

# Validate inputs
if '-' in OANDA_ACCOUNT_ID and len(OANDA_ACCOUNT_ID) > 50:
    print("\n⚠️ WARNING: Account ID looks like a token - you may have swapped them!")
    print("   Swapping automatically...")
    OANDA_API_TOKEN, OANDA_ACCOUNT_ID = OANDA_ACCOUNT_ID, OANDA_API_TOKEN

# Set environment variables
os.environ["OANDA_API_TOKEN"] = OANDA_API_TOKEN
os.environ["OANDA_ACCOUNT_ID"] = OANDA_ACCOUNT_ID

# Verify (show only last 4 chars of token)
print(f"\n✅ OANDA_API_TOKEN: ...{OANDA_API_TOKEN[-4:]}")
print(f"✅ OANDA_ACCOUNT_ID: {OANDA_ACCOUNT_ID}")

🔐 OANDA API Configuration

Enter your OANDA Practice account credentials.
(These are stored only in this session's memory)

API Token format: xxxx-xxxx (long string with hyphen)

Account ID format: 101-001-XXXXXXXX-001

✅ OANDA_API_TOKEN: ...1a62
✅ OANDA_ACCOUNT_ID: 101-001-37949116-001


In [15]:
# ============================================================
# 3.2 Test OANDA connection
# ============================================================
import sys
sys.path.insert(0, '/content/ml_engine')

try:
    from oanda_practice import OandaPracticeClient
    
    client = OandaPracticeClient.from_env()
    print("✅ OANDA client initialized successfully!")
    print("\n📊 Testing candle fetch...")
    
    # Fetch a small sample to verify connection
    resp = client.get_candles(
        instrument="EUR_USD",
        granularity="H1",
        count=10
    )
    # Response is a dict with "candles" key
    candles = resp.get("candles", []) if isinstance(resp, dict) else []
    
    print(f"✅ Fetched {len(candles)} candles from OANDA")
    if candles:
        c = candles[-1]
        print(f"   Latest: {c['time']} Close: {c['mid']['c']}")
    
except Exception as e:
    print(f"❌ OANDA connection failed: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️ You can still train using local CSV files.")
    print("   Upload your market data to /content/ml_engine/market_data/")

✅ OANDA client initialized successfully!

📊 Testing candle fetch...
✅ Fetched 10 candles from OANDA
   Latest: 2026-01-06T16:00:00.000000000Z Close: 1.16877


## 4️⃣ Data Preparation

In [ ]:
# ============================================================
# 4.1 Fetch 12,000 candles for training
# ============================================================
import pandas as pd
from datetime import datetime

# Configuration
INSTRUMENT = "EUR_USD"  # Change to your preferred pair
GRANULARITY = "H1"      # H1 = 1 hour candles
TARGET_CANDLES = 12000  # ~500 days of H1 data

print(f"📊 Fetching {TARGET_CANDLES} {GRANULARITY} candles for {INSTRUMENT}...")
print(f"   This will take a few minutes for large requests...\n")

all_candles = []
from_time = None

# OANDA limits to 5000 per request, so we fetch in batches
while len(all_candles) < TARGET_CANDLES:
    remaining = TARGET_CANDLES - len(all_candles)
    batch_size = min(5000, remaining)
    
    try:
        if from_time:
            resp = client.get_candles(
                instrument=INSTRUMENT,
                granularity=GRANULARITY,
                count=batch_size,
                to_time=from_time  # Fetch older candles
            )
        else:
            resp = client.get_candles(
                instrument=INSTRUMENT,
                granularity=GRANULARITY,
                count=batch_size
            )
        
        # Extract candles from response dict
        candles = resp.get("candles", []) if isinstance(resp, dict) else []
        
        if not candles:
            print(f"   No more candles available")
            break
            
        all_candles = candles + all_candles  # Prepend older candles
        from_time = candles[0]['time']  # Get timestamp of oldest candle
        
        print(f"   Fetched batch: {len(candles)} candles | Total: {len(all_candles)}")
        
    except Exception as e:
        print(f"   ⚠️ Batch fetch error: {e}")
        break

# Convert to DataFrame and save
DATA_PATH = None
CANDLES = 0

if all_candles:
    df = pd.DataFrame([{
        'time': c['time'],
        'open': float(c['mid']['o']),
        'high': float(c['mid']['h']),
        'low': float(c['mid']['l']),
        'close': float(c['mid']['c']),
        'volume': int(c['volume'])
    } for c in all_candles])
    
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.sort_index(inplace=True)
    
    # Save to market_data
    DATA_PATH = f"/content/ml_engine/market_data/{INSTRUMENT.replace('_', '')}_{GRANULARITY}.csv"
    df.to_csv(DATA_PATH)
    CANDLES = len(df)
    
    print(f"\n✅ Saved {CANDLES} candles to {DATA_PATH}")
    print(f"   Date range: {df.index[0]} to {df.index[-1]}")
    print(f"\n📊 Data Preview:")
    display(df.tail(5))
else:
    print("❌ No candles fetched. Check your OANDA credentials and connection.")

📊 Fetching 12000 H1 candles for EUR_USD...
   This will take a few minutes for large requests...

   Fetched batch: 5000 candles | Total: 5000
   Fetched batch: 5000 candles | Total: 10000
   Fetched batch: 2000 candles | Total: 12000

✅ Saved 12000 candles to /content/ml_engine/market_data/EURUSD_H1.csv
   Date range: 2024-01-31 15:00:00+00:00 to 2026-01-06 16:00:00+00:00

📊 Data Preview:


,open,high,low,close,volume
time,,,,,
2026-01-06 12:00:00+00:00,1.17062,1.17122,1.16972,1.17014,5267
2026-01-06 13:00:00+00:00,1.17014,1.17137,1.16975,1.17110,7498
2026-01-06 14:00:00+00:00,1.17110,1.17176,1.16992,1.17092,8688
2026-01-06 15:00:00+00:00,1.17092,1.17092,1.16840,1.16953,10993
2026-01-06 16:00:00+00:00,1.16953,1.16984,1.16860,1.16874,4232


In [16]:
# ============================================================
# 4.2 Data preview and validation
# ============================================================
import pandas as pd

# Check if DATA_PATH was set by previous cell
if 'DATA_PATH' not in dir() or DATA_PATH is None:
    # Try to find existing data file
    import glob
    csv_files = glob.glob("/content/ml_engine/market_data/*.csv")
    if csv_files:
        DATA_PATH = csv_files[0]
        print(f"📁 Using existing data: {DATA_PATH}")
    else:
        print("❌ No data file found. Run cell 4.1 first to fetch OANDA data,")
        print("   or upload a CSV to /content/ml_engine/market_data/")
        DATA_PATH = None

if DATA_PATH:
    df = pd.read_csv(DATA_PATH)
    CANDLES = len(df)
    
    print("=" * 70)
    print("📊 Data Preview")
    print("=" * 70)
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nLast 5 rows:")
    display(df.tail())
    print(f"\nStatistics:")
    display(df.describe())
    
    # Check for NaN values
    nan_counts = df.isna().sum()
    if nan_counts.any():
        print(f"\n⚠️ NaN values detected:")
        print(nan_counts[nan_counts > 0])
    else:
        print(f"\n✅ No NaN values in data")

📁 Using existing data: /content/ml_engine/market_data/oanda_EUR_USD_H1_live_20260104_154229.csv
📊 Data Preview
Shape: (5000, 9)

Columns: ['time', 'open', 'high', 'low', 'close', 'volume', 'bid_close', 'ask_close', 'close_ema_14']

First 5 rows:


,time,open,high,low,close,volume,bid_close,ask_close,close_ema_14
0,2025-03-14T13:00:00Z,1.08946,1.08981,1.08703,1.08744,11753.0,1.08734,1.08755,1.087440
1,2025-03-14T14:00:00Z,1.08744,1.08866,1.08656,1.08780,14382.0,1.08772,1.08788,1.087488
2,2025-03-14T15:00:00Z,1.08779,1.08868,1.08669,1.08715,12119.0,1.08708,1.08722,1.087443
3,2025-03-14T16:00:00Z,1.08716,1.08900,1.08704,1.08831,7436.0,1.08823,1.08839,1.087559
4,2025-03-14T17:00:00Z,1.08832,1.08881,1.08791,1.08824,5268.0,1.08816,1.08831,1.087649



Last 5 rows:


,time,open,high,low,close,volume,bid_close,ask_close,close_ema_14
4995,2026-01-02T17:00:00Z,1.17374,1.17376,1.17286,1.17308,4792.0,1.17300,1.17316,1.173501
4996,2026-01-02T18:00:00Z,1.17308,1.17314,1.17154,1.17192,4792.0,1.17184,1.17199,1.173290
4997,2026-01-02T19:00:00Z,1.17191,1.17244,1.17144,1.17232,4435.0,1.17224,1.17241,1.173161
4998,2026-01-02T20:00:00Z,1.17233,1.17238,1.17153,1.17178,3253.0,1.17169,1.17186,1.172977
4999,2026-01-02T21:00:00Z,1.17176,1.17234,1.17174,1.17199,1824.0,1.17187,1.17211,1.172845



Statistics:


,open,high,low,close,volume,bid_close,ask_close,close_ema_14
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,1.151688,1.152460,1.150951,1.151708,6715.677400,1.151620,1.151797,1.151597
std,0.025463,0.025316,0.025586,0.025446,5454.599276,0.025447,0.025445,0.025458
min,1.073700,1.074460,1.073300,1.073710,148.000000,1.073560,1.073810,1.076044
25%,1.138777,1.139820,1.138008,1.138823,3306.000000,1.138683,1.138968,1.138679
50%,1.160370,1.160980,1.159800,1.160380,5444.500000,1.160300,1.160465,1.160592
75%,1.169852,1.170523,1.169160,1.169885,8499.000000,1.169777,1.169987,1.169702
max,1.187110,1.191880,1.186600,1.187100,59068.000000,1.187030,1.187180,1.185283



✅ No NaN values in data


## 5️⃣ Training Configuration (CUDA-Optimized)

In [48]:
# ============================================================
# 5.1 Training hyperparameters - FULLY OPTIMIZED for A100 80GB
# ============================================================
# Based on NVIDIA A100 Tensor Core & TensorFlow Mixed Precision guidelines

# === A100 KEY CAPABILITIES ===
# • 80GB HBM2e VRAM (2TB/s bandwidth)
# • 312 TFLOPS FP16 Tensor Core performance
# • 156 TFLOPS TF32 performance  
# • 19.5 TFLOPS FP32 performance
# • TF32 enabled by default (faster than FP32, same accuracy)
# • Full FP16/BF16 Tensor Core support

# === TENSOR CORE OPTIMIZATION RULES ===
# All dimensions must be MULTIPLES OF 8 for maximum Tensor Core utilization:
# • batch_size: multiple of 8 (ideally 64, 128, 256, 512)
# • d_model: multiple of 8 (32, 64, 128, 256...)
# • num_heads: multiple of 8 or divisor of d_model
# • dff: multiple of 8 (64, 128, 256...)
# • Dense layer units: multiple of 8

TRAINING_CONFIG = {
    # === MODEL ARCHITECTURE (Tensor Core Optimized) ===
    "model_type": "ensemble",  # Modular ensemble: Transformer + XGBoost + RF + Ridge
    
    # Transformer (Direction Predictor) - ALL DIMS MULTIPLE OF 8
    "transformer_d_model": 64,      # Increased from 32 → 64 (multiple of 8)
    "transformer_num_heads": 8,     # Increased from 4 → 8 (multiple of 8)
    "transformer_num_layers": 2,    # Keep same (depth doesn't affect Tensor Cores)
    "transformer_dff": 128,         # Increased from 64 → 128 (multiple of 8)
    "transformer_dropout": 0.2,
    
    # === A100-OPTIMIZED TRAINING PARAMETERS ===
    "epochs": 200,
    "batch_size": 512,              # ⬆️ A100 80GB can handle 512 easily (multiple of 8)
    "learning_rate": 0.0005,        # Slightly higher for larger batch (sqrt scaling)
    "patience": 20,                 # Early stopping patience
    "seq_len": 64,                  # ⬆️ Increased from 60 → 64 (multiple of 8!)
    
    # === A100-SPECIFIC OPTIMIZATIONS ===
    "mixed_precision": True,        # float16 compute (up to 2x speedup)
    "use_tf32": True,               # TF32 for float32 ops (1.5x speedup, A100 only)
    "jit_compile": True,            # XLA compilation (additional speedup)
    "steps_per_execution": 32,      # ⬆️ Higher for A100 (reduce Python overhead)
    
    # === MEMORY OPTIMIZATION ===
    "prefetch_buffer": 4,           # tf.data prefetch buffer
    "num_parallel_calls": 8,        # tf.data parallel calls
    
    # === CONTINUAL LEARNING (IDENTICAL TO MAC) ===
    "use_ema": True,                # Exponential Moving Average
    "ema_decay": 0.999,
    "use_ewc": True,                # Elastic Weight Consolidation
    "ewc_lambda": 1000.0,
    "use_replay_buffer": True,
    "replay_buffer_ratio": 0.10,
    
    # === WALK-FORWARD VALIDATION ===
    "cv_folds": 3,                  # Walk-forward cross-validation
    "min_train_samples": 4000,
    "test_period": 1000,
    "gap": 24,                      # 1 day gap to prevent leakage
    
    # === OVERFITTING PREVENTION ===
    "enable_swa": True,             # Stochastic Weight Averaging
    "enable_cosine_restarts": True,
    "overfit_threshold": 0.08,
    "critical_threshold": 0.15,
    "max_acceptable_gap": 0.12,
}

print("=" * 70)
print("⚙️ Training Configuration (A100 80GB FULLY OPTIMIZED)")
print("=" * 70)
print("\n🚀 A100 Tensor Core Optimizations Applied:")
print("   ┌─────────────────────────────────────────────────────────────┐")
print("   │ Parameter          │ Before    │ After     │ Reason        │")
print("   ├─────────────────────────────────────────────────────────────┤")
print("   │ batch_size         │ 256       │ 512       │ 80GB VRAM     │")
print("   │ d_model            │ 32        │ 64        │ Multiple of 8 │")
print("   │ num_heads          │ 4         │ 8         │ Multiple of 8 │")
print("   │ dff                │ 64        │ 128       │ Multiple of 8 │")
print("   │ seq_len            │ 60        │ 64        │ Multiple of 8 │")
print("   │ steps_per_execution│ 20        │ 32        │ Less overhead │")
print("   │ learning_rate      │ 0.0003    │ 0.0005    │ sqrt(batch)   │")
print("   └─────────────────────────────────────────────────────────────┘")
print("\n📊 Expected Performance Gains:")
print("   • Mixed Precision (FP16): ~2x speedup on Tensor Cores")
print("   • TF32 for FP32 ops: ~1.5x speedup (A100 exclusive)")
print("   • XLA Compilation: ~1.2-1.5x additional speedup")
print("   • Larger batch: Better GPU utilization")
print("\n✅ All tensor dimensions are multiples of 8 for Tensor Core acceleration!")

⚙️ Training Configuration (A100 80GB FULLY OPTIMIZED)

🚀 A100 Tensor Core Optimizations Applied:
   ┌─────────────────────────────────────────────────────────────┐
   │ Parameter          │ Before    │ After     │ Reason        │
   ├─────────────────────────────────────────────────────────────┤
   │ batch_size         │ 256       │ 512       │ 80GB VRAM     │
   │ d_model            │ 32        │ 64        │ Multiple of 8 │
   │ num_heads          │ 4         │ 8         │ Multiple of 8 │
   │ dff                │ 64        │ 128       │ Multiple of 8 │
   │ seq_len            │ 60        │ 64        │ Multiple of 8 │
   │ steps_per_execution│ 20        │ 32        │ Less overhead │
   │ learning_rate      │ 0.0003    │ 0.0005    │ sqrt(batch)   │
   └─────────────────────────────────────────────────────────────┘

📊 Expected Performance Gains:
   • Mixed Precision (FP16): ~2x speedup on Tensor Cores
   • TF32 for FP32 ops: ~1.5x speedup (A100 exclusive)
   • XLA Compilation: ~1.2-1.5x

## 6️⃣ Run Training

In [55]:
# ============================================================
# 6.1 Import training modules
# ============================================================
import os
import sys
import logging

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Add repo to path
sys.path.insert(0, '/content/ml_engine')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("📦 Importing training modules...")

import numpy as np
import pandas as pd
import tensorflow as tf

# Import trainers (correct class names!)
from modular_trainers import (
    TrainerConfig,
    TransformerDirectionTrainer,
    XGBoostTrainer,          # NOT XGBoostMomentumTrainer
    RandomForestTrainer,     # NOT RandomForestRiskTrainer
    RidgeTrainer,            # NOT RidgeConfidenceTrainer
    OverfitPreventionCallback,
)

# Import data loaders (correct function names!)
from modular_data_loaders import (
    compute_normalized_features,
    load_direction_data,      # NOT prepare_direction_data
    load_xgboost_data,        # NOT prepare_momentum_data
    load_rf_data,             # NOT prepare_risk_data
    load_ridge_data,          # NOT prepare_confidence_data
)

# Import feature engineering
from feature_engineering import FeatureEngineering

print("✅ All modules imported successfully!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   GPU devices: {tf.config.list_physical_devices('GPU')}")

📦 Importing training modules...
✅ All modules imported successfully!
   TensorFlow: 2.19.0
   GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [56]:
# ============================================================
# 6.2 Prepare training data
# ============================================================
from rich.console import Console
from rich.panel import Panel

console = Console()

console.print(Panel("[bold blue]Step 1: Data Preparation[/bold blue]"))

# Load data
df = pd.read_csv(DATA_PATH)
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

# Rename columns to lowercase
df.columns = [c.lower() for c in df.columns]

console.print(f"  📊 Loaded {len(df)} candles")
console.print(f"  📅 Date range: {df.index.min()} to {df.index.max()}")

# Compute normalized features
console.print("  🔧 Computing normalized features...")
df = compute_normalized_features(df)

# Add technical indicators
fe = FeatureEngineering()
df = fe.add_technical_indicators(df)

# Fill NaN values
df = df.ffill().bfill()

# Drop remaining NaN rows
df = df.dropna()

console.print(f"  ✅ Features computed: {len(df.columns)} columns")
console.print(f"  ✅ Clean rows: {len(df)}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 1: Data Preparation                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Loaded 5000 candles

📅 Date range: 2025-03-14 13:00:00+00:00 to 2026-01-02 21:00:00+00:00

🔧 Computing normalized features...

✅ Features computed: 120 columns

✅ Clean rows: 5000

In [57]:
# ============================================================
# 6.3 Prepare model-specific datasets
# ============================================================
console.print(Panel("[bold blue]Step 2: Prepare Model-Specific Data[/bold blue]"))

SEQ_LEN = TRAINING_CONFIG["seq_len"]

# Direction data (Transformer) - uses load_direction_data
# IMPORTANT: Use threshold=0.0 to include ALL samples (no filtering)
# This prevents class imbalance from threshold-based filtering
console.print("  📊 Preparing Direction data (Transformer)...")
direction_data = load_direction_data(
    df, 
    split=(0.8, 0.1, 0.1), 
    lookahead=6,
    threshold=0.0  # <-- KEY FIX: Include all samples, no filtering
)
console.print(f"     X_train: {direction_data['X_train'].shape}")
console.print(f"     y_train: {direction_data['y_train'].shape}")

# Check class distribution
y_train_direction = direction_data['y_train']
n_up = (y_train_direction == 1).sum()
n_down = (y_train_direction == 0).sum()
n_unclear = ((y_train_direction != 0) & (y_train_direction != 1)).sum()
total = len(y_train_direction)

console.print(f"     📊 Class distribution: UP={n_up} ({100*n_up/total:.1f}%), DOWN={n_down} ({100*n_down/total:.1f}%)")
if n_unclear > 0:
    console.print(f"     ⚠️ Unclear samples: {n_unclear} ({100*n_unclear/total:.1f}%)")

# Check imbalance ratio
if n_up > 0 and n_down > 0:
    imbalance = max(n_up, n_down) / min(n_up, n_down)
    if imbalance > 2.0:
        console.print(f"     ⚠️ High imbalance: {imbalance:.2f}x - class weights will be applied")
    else:
        console.print(f"     ✅ Balanced classes (imbalance: {imbalance:.2f}x)")

# Momentum data (XGBoost) - uses load_xgboost_data
console.print("  📊 Preparing Momentum data (XGBoost)...")
momentum_data = load_xgboost_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {momentum_data['X_train'].shape}")

# Risk data (Random Forest) - uses load_rf_data
console.print("  📊 Preparing Risk data (Random Forest)...")
risk_data = load_rf_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {risk_data['X_train'].shape}")

# Confidence data (Ridge) - uses load_ridge_data
console.print("  📊 Preparing Confidence data (Ridge)...")
confidence_data = load_ridge_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {confidence_data['X_train'].shape}")

console.print("\n✅ All datasets prepared!")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 2: Prepare Model-Specific Data                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Preparing Direction data (Transformer)...

X_train: (3995, 49)

y_train: (3995,)

📊 Class distribution: UP=2073 (51.9%), DOWN=1922 (48.1%)

...

📊 Preparing Risk data (Random Forest)...

X_train: (3992, 18)

📊 Preparing Confidence data (Ridge)...

X_train: (3992, 14)

✅ All datasets prepared!

In [ ]:
# ============================================================
# 6.4 Train Transformer (Direction Predictor)
# ============================================================
console.print(Panel("[bold green]Step 3/6: Training Transformer (Direction)[/bold green]"))

# Create trainer config
config = TrainerConfig(
    epochs=TRAINING_CONFIG["epochs"],
    batch_size=TRAINING_CONFIG["batch_size"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    patience=TRAINING_CONFIG["patience"],
    transformer_d_model=TRAINING_CONFIG["transformer_d_model"],
    transformer_num_heads=TRAINING_CONFIG["transformer_num_heads"],
    transformer_num_layers=TRAINING_CONFIG["transformer_num_layers"],
    transformer_dff=TRAINING_CONFIG["transformer_dff"],
    transformer_dropout=TRAINING_CONFIG["transformer_dropout"],
    use_ema=TRAINING_CONFIG["use_ema"],
    ema_decay=TRAINING_CONFIG["ema_decay"],
    use_ewc=TRAINING_CONFIG["use_ewc"],
    ewc_lambda=TRAINING_CONFIG["ewc_lambda"],
)

# Create and train Transformer
transformer_trainer = TransformerDirectionTrainer(config)

console.print(f"  🏗️ Building Transformer model...")
console.print(f"     d_model={config.transformer_d_model}, heads={config.transformer_num_heads}")
console.print(f"     layers={config.transformer_num_layers}, dff={config.transformer_dff}")

# Train - method signature: train(X_train, y_train, X_val, y_val, feature_names, w_train, w_val, warm_start_path, instrument, data_range)
# SWA and cosine restarts are configured INSIDE the trainer via OverfitPreventionCallback
transformer_result = transformer_trainer.train(
    X_train=direction_data['X_train'],
    y_train=direction_data['y_train'],
    X_val=direction_data['X_val'],
    y_val=direction_data['y_val'],
    feature_names=direction_data.get('feature_names'),
    instrument=INSTRUMENT,
    data_range=f"{TARGET_CANDLES} candles",
)

console.print(f"\n✅ Transformer trained!")
console.print(f"   Val Accuracy: {transformer_result.get('val_accuracy', 0):.1%}")
# FIX: Use correct key 'val_balanced_accuracy' (not 'balanced_accuracy')
console.print(f"   Balanced Acc: {transformer_result.get('val_balanced_accuracy', 0):.1%}")
console.print(f"   ↳ UP accuracy:   {transformer_result.get('val_up_accuracy', 0):.1%}")
console.print(f"   ↳ DOWN accuracy: {transformer_result.get('val_down_accuracy', 0):.1%}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 3/6: Training Transformer (Direction)                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🏗️ Building Transformer model...

d_model=64, heads=8

layers=2, dff=128

Training Transformer Direction for up to 200 epochs...

...

📊 Gap stats: min=-1.2%, avg=3.3%, max=6.5%

🔄 Warm restarts: 3

✅ Transformer trained!

Val Accuracy: 51.9%

Balanced Acc: 0.0%

In [59]:
# ============================================================
# 6.5 Train XGBoost (Momentum Analyzer)
# ============================================================
console.print(Panel("[bold green]Step 4/6: Training XGBoost (Momentum)[/bold green]"))

xgb_trainer = XGBoostTrainer(config)

xgb_result = xgb_trainer.train(
    X_train=momentum_data['X_train'],
    y_train=momentum_data['y_train'],
    X_val=momentum_data['X_val'],
    y_val=momentum_data['y_val'],
    feature_names=momentum_data.get('feature_names'),
)

console.print(f"\n✅ XGBoost trained!")
console.print(f"   Momentum MAE: {xgb_result.get('momentum_mae', 0):.4f}")
console.print(f"   Accel Accuracy: {xgb_result.get('acceleration_accuracy', 0):.1%}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 4/6: Training XGBoost (Momentum)                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ XGBoost trained!

Momentum MAE: 0.0268

Accel Accuracy: 88.6%

In [60]:
# ============================================================
# 6.6 Train Random Forest (Risk Assessor)
# ============================================================
console.print(Panel("[bold green]Step 5/6: Training Random Forest (Risk)[/bold green]"))

rf_trainer = RandomForestTrainer(config)

rf_result = rf_trainer.train(
    X_train=risk_data['X_train'],
    y_train=risk_data['y_train'],
    X_val=risk_data['X_val'],
    y_val=risk_data['y_val'],
    feature_names=risk_data.get('feature_names'),
)

console.print(f"\n✅ Random Forest trained!")
console.print(f"   Drawdown MAE: {rf_result.get('drawdown_mae', 0):.4f}")
console.print(f"   Streak MAE: {rf_result.get('streak_mae', 0):.4f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 5/6: Training Random Forest (Risk)                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Random Forest trained!

Drawdown MAE: 0.0000

Streak MAE: 0.0000

In [61]:
# ============================================================
# 6.7 Train Ridge (Confidence Scorer)
# ============================================================
console.print(Panel("[bold green]Step 6/6: Training Ridge (Confidence)[/bold green]"))

ridge_trainer = RidgeTrainer(config)

ridge_result = ridge_trainer.train(
    X_train=confidence_data['X_train'],
    y_train=confidence_data['y_train'],
    X_val=confidence_data['X_val'],
    y_val=confidence_data['y_val'],
    feature_names=confidence_data.get('feature_names'),
)

console.print(f"\n✅ Ridge trained!")
console.print(f"   Confidence MAE: {ridge_result.get('confidence_mae', 0):.2f}")
console.print(f"   R² Score: {ridge_result.get('r2_score', 0):.3f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 6/6: Training Ridge (Confidence)                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Ridge trained!

Confidence MAE: 2.11

R² Score: -323264.836

In [62]:
# ============================================================
# 6.8 Save all models
# ============================================================
console.print(Panel("[bold blue]Saving Models[/bold blue]"))

import json
from datetime import datetime
import shutil

MODEL_DIR = "trained_data/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save Transformer (with instrument for replay buffer)
transformer_trainer.save(f"{MODEL_DIR}/transformer_direction.keras", instrument=INSTRUMENT)
console.print(f"  💾 Saved: {MODEL_DIR}/transformer_direction.keras")

# Save XGBoost
xgb_trainer.save(f"{MODEL_DIR}/xgb_momentum.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/xgb_momentum.pkl")

# Save Random Forest
rf_trainer.save(f"{MODEL_DIR}/rf_risk.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/rf_risk.pkl")

# Save Ridge
ridge_trainer.save(f"{MODEL_DIR}/ridge_confidence.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/ridge_confidence.pkl")

# Save ensemble metadata
metadata = {
    "trained_at": datetime.now().isoformat(),
    "trained_on": "colab_cuda_a100",
    "instrument": INSTRUMENT,
    "granularity": GRANULARITY,
    "candles": CANDLES,
    "config": TRAINING_CONFIG,
    "results": {
        "transformer": transformer_result,
        "xgboost": xgb_result,
        "random_forest": rf_result,
        "ridge": ridge_result,
    }
}

with open(f"{MODEL_DIR}/modular_ensemble.meta.json", 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
console.print(f"  💾 Saved: {MODEL_DIR}/modular_ensemble.meta.json")

console.print("\n✅ All models saved!")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Saving Models                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

💾 Saved: trained_data/models/transformer_direction.keras

💾 Saved: trained_data/models/xgb_momentum.pkl

💾 Saved: trained_data/models/rf_risk.pkl

💾 Saved: trained_data/models/ridge_confidence.pkl

💾 Saved: trained_data/models/modular_ensemble.meta.json

✅ All models saved!

## 7️⃣ Training Summary & Visualization

In [63]:
# ============================================================
# 7.1 Training summary
# ============================================================
from rich.table import Table

console.print(Panel("[bold green]🎉 Training Complete![/bold green]"))

summary_table = Table(title="Model Performance Summary")
summary_table.add_column("Model", style="cyan")
summary_table.add_column("Metric", style="magenta")
summary_table.add_column("Value", style="green")

summary_table.add_row("Transformer", "Val Accuracy", f"{transformer_result.get('val_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "Balanced Acc", f"{transformer_result.get('balanced_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Accel Accuracy", f"{xgb_result.get('accel_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Momentum MAE", f"{xgb_result.get('momentum_mae', 0):.4f}")
summary_table.add_row("Random Forest", "Drawdown MAE", f"{rf_result.get('drawdown_mae', 0):.4f}")
summary_table.add_row("Random Forest", "Streak MAE", f"{rf_result.get('streak_mae', 0):.4f}")
summary_table.add_row("Ridge", "R² Score", f"{ridge_result.get('r2_score', 0):.3f}")
summary_table.add_row("Ridge", "Confidence MAE", f"{ridge_result.get('confidence_mae', 0):.2f}")

console.print(summary_table)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🎉 Training Complete!                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           Model Performance Summary            
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Model         ┃ Metric         ┃ Value       ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Transformer   │ Val Accuracy   │ 51.9%       │
│ Transformer   │ Balanced Acc   │ 0.0%        │
│ XGBoost       │ Accel Accuracy │ 0.0%        │
│ XGBoost       │ Momentum MAE   │ 0.0268      │
│ Random Forest │ Drawdown MAE   │ 0.0000      │
│ Random Forest │ Streak MAE     │ 0.0000      │
│ Ridge         │ R² Score       │ -323264.836 │
│ Ridge         │ Confidence MAE │ 2.11        │
└───────────────┴────────────────┴─────────────┘

## 8️⃣ Download Models

In [64]:
# ============================================================
# 8.1 Package models for download
# ============================================================
import shutil
from datetime import datetime

# Create zip file with all models
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f"ml_engine_models_{INSTRUMENT.replace('/', '_')}_{timestamp}"

# Create archive
shutil.make_archive(
    f"/content/{zip_name}",
    'zip',
    root_dir='/content/ml_engine',
    base_dir='trained_data/models'
)

print(f"✅ Models packaged: /content/{zip_name}.zip")
print(f"\n📦 Contents:")
!unzip -l /content/{zip_name}.zip | head -20

✅ Models packaged: /content/ml_engine_models_EUR_USD_20260106_173007.zip

📦 Contents:
Archive:  /content/ml_engine_models_EUR_USD_20260106_173007.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-01-06 17:26   trained_data/models/
        0  2026-01-06 17:26   trained_data/models/checkpoints/
     1901  2026-01-06 17:30   trained_data/models/modular_ensemble.meta.json
      910  2026-01-06 17:26   trained_data/models/training_report_20260105_233453.md
   994838  2026-01-06 17:30   trained_data/models/xgb_momentum.pkl
     4318  2026-01-06 17:30   trained_data/models/transformer_direction.meta.pkl
     2201  2026-01-06 17:30   trained_data/models/ridge_confidence.pkl
      909  2026-01-06 17:26   trained_data/models/training_report_20260106_014935.md
   284222  2026-01-06 17:30   trained_data/models/transformer_direction.ema.pkl
   568280  2026-01-06 17:30   trained_data/models/transformer_direction.ewc.pkl
  1068061  2026-01-06 17:30   trained_da

In [65]:
# ============================================================
# 8.2 Download to local machine
# ============================================================
from google.colab import files

print("📥 Downloading models to your local machine...")
print("   (This will open a download dialog)\n")

files.download(f"/content/{zip_name}.zip")

print("\n✅ Download started!")
print("\n📋 To use on your Mac:")
print("   1. Unzip the downloaded file")
print("   2. Copy contents to ml_engine/trained_data/models/")
print("   3. Run: buddy analyze --model-type ensemble")

📥 Downloading models to your local machine...
   (This will open a download dialog)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started!

📋 To use on your Mac:
   1. Unzip the downloaded file
   2. Copy contents to ml_engine/trained_data/models/
   3. Run: buddy analyze --model-type ensemble


## 9️⃣ Optional: Push to GitHub

In [66]:
# ============================================================
# 9.1 Commit and push trained models to GitHub
# ============================================================
# ⚠️ Only run this if you want to push models to your repo

PUSH_TO_GITHUB = False  # Set to True to enable

if PUSH_TO_GITHUB:
    from getpass import getpass
    
    print("🔐 GitHub Authentication")
    print("Enter your GitHub Personal Access Token (PAT)")
    print("Create one at: https://github.com/settings/tokens\n")
    
    GITHUB_TOKEN = getpass("GitHub PAT: ")
    GITHUB_USER = input("GitHub Username: ")
    GITHUB_EMAIL = input("GitHub Email: ")
    
    # Configure git
    !git config --global user.name "{GITHUB_USER}"
    !git config --global user.email "{GITHUB_EMAIL}"
    
    # Set remote with token
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Raynergy-svg/ml_engine.git
    
    # Add and commit
    !git add trained_data/models/
    !git commit -m "feat: Add CUDA-trained models from Colab ({INSTRUMENT})"
    
    # Push
    !git push origin main
    
    print("\n✅ Models pushed to GitHub!")
else:
    print("ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.")

ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.


---

## 📝 Notes

### GPU Memory Usage
- T4 (16GB): Handles batch_size=128 comfortably
- A100 (40GB): Can increase batch_size to 256-512

### Training Time Estimates
- T4 GPU: ~15-20 minutes for full ensemble
- A100 GPU: ~5-10 minutes

### Differences from Mac (Metal)
- Using CUDA instead of Metal
- XLA compilation enabled (jit_compile=True)
- Same model architecture and hyperparameters

### Troubleshooting
- OOM Error: Reduce batch_size to 64
- OANDA timeout: Fetch smaller batches
- Import errors: Restart runtime and re-run setup cells